In [1]:
!nvidia-smi

Sat Mar 14 15:00:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
# Install required libraries
!pip install datasets transformers huggingface_hub
!apt-get install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [4]:
from datasets import load_dataset

imdb = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
import pandas as pd

#Convert to pandas DataFrame
train_df = pd.DataFrame(imdb['train'])
test_df = pd.DataFrame(imdb['test'])

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

In [7]:
model_name = "distilbert-base-uncased"

In [8]:
train_df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [9]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)       # remove URLs
    text = re.sub(r"<.*?>", "", text)         # remove HTML
    text = re.sub(r"[^a-zA-Z\s]", "", text)   # remove special chars
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces

    return text


train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df["token_length"] = train_df["text"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (554 > 512). Running this sequence through the model will result in indexing errors


In [11]:
train_df = train_df[train_df["token_length"] < 512]

In [12]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22624 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   text          22624 non-null  object
 1   label         22624 non-null  int64 
 2   token_length  22624 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 707.0+ KB


In [13]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [14]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256        # 512 is used for capture more context but train time will be high, for fast training use 256
    )
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/22624 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [15]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [18]:
training_args = TrainingArguments(

    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="epoch"
)

In [19]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

In [20]:
trainer.train()

Step,Training Loss
500,0.337792
1000,0.282388
1500,0.241217
2000,0.186942
2500,0.172478
3000,0.147789
3500,0.111433
4000,0.113312
4500,0.092235
5000,0.046218


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14140, training_loss=0.07384644257697728, metrics={'train_runtime': 5334.1687, 'train_samples_per_second': 42.413, 'train_steps_per_second': 2.651, 'total_flos': 1.498471213596672e+16, 'train_loss': 0.07384644257697728, 'epoch': 10.0})

In [21]:
import numpy as np

In [22]:
results = trainer.evaluate()

print("\nEvaluation Results")
print(results)


Evaluation Results
{'eval_loss': 0.7527818083763123, 'eval_accuracy': 0.90808, 'eval_precision': 0.9000156838143036, 'eval_recall': 0.91816, 'eval_f1': 0.9089973071439886, 'eval_runtime': 184.4155, 'eval_samples_per_second': 135.563, 'eval_steps_per_second': 8.475, 'epoch': 10.0}


In [23]:
trainer.save_model("./model")

tokenizer.save_pretrained("./model")

print("\nModel saved to sentiment_model/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to sentiment_model/


In [24]:
tokenizer = AutoTokenizer.from_pretrained("model")
model = AutoModelForSequenceClassification.from_pretrained("model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [27]:
def predict(text):
        """
        Predict sentiment for a single text input.

        Args:
            text (str): Input text to classify

        Returns:
            dict: Prediction result containing:
                - "text": input text
                - "sentiment": "positive" or "negative"
                - "confidence": probability of predicted class (0.00–1.00)
        """
        cleaned_text = clean_text(text)

        inputs = tokenizer(
            cleaned_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        )

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)
            predicted_class = torch.argmax(probabilities).item()
            confidence = probabilities[0][predicted_class].item()

        label = "positive" if predicted_class == 1 else "negative"

        return {
            "text":text,
            "sentiment": label,
            "confidence": round(confidence, 2)
        }

In [33]:
predict("It was boring at first but it was fine at end")

{'text': 'It was boring at first but it was fine at end',
 'sentiment': 'negative',
 'confidence': 0.98}